# Singapore, Connected | Exploratory Data Analysis & Data Processing

**Data Source**  
UN International Migrant Stock 2024 estimates of migrant populations by country or area of origin and destination, including regional aggregates.

**Focus of EDA**
Origins of migrants living in Singapore and destinations of migrants originating from Singapore at mid-2024.

--- 
### Step 1: Inspect the Workbook
Identify the available worksheets and locate the origin <-> destination data.

In [ ]:
from pathlib import Path
import pandas as pd

# Load the original UN workbook as is
data_path = Path("data/migrantstock_raw.xlsx")
workbook = pd.ExcelFile(data_path)

# Display sheet names
workbook.sheet_names


['Table of contents',
 'Table 1',
 'Table 2',
 'Table 3',
 'Table 4',
 'Table 5',
 'Table 6',
 'Table 7',
 'Table 8',
 'Table 9',
 'Migrant notes']

In [ ]:
# Explore contents of second sheet
df = pd.read_excel(data_path, sheet_name="Table 1", header=10)

df.head()

,Index,"Region, development group, country or area of destination",Coverage,Data type,Location code of destination,"Region, development group, country or area of origin",Location code of origin,1990,1995,2000,...,2020.1,2024.1,1990.2,1995.2,2000.2,2005.2,2010.2,2015.2,2020.2,2024.2
0,1,World,NaN,NaN,900,World,900,153916063.0,163176002,174566152,...,143223497,158009795,76143981,80632704,86275210,94338056,107763606,119854396,132060535,146012018
1,2,World,NaN,NaN,900,Sub-Saharan Africa,1834,14124662.0,15183742,14584913,...,14375255,16046945,6665704,7163563,6846928,7325563,8503651,10696751,12759702,14614665
2,3,World,NaN,NaN,900,Northern Africa and Western Asia,1833,14986109.0,17216219,18728264,...,21212741,22913639,6648687,7547605,8080447,9029322,10800712,13724647,15984112,17615687
3,4,World,NaN,NaN,900,Central and Southern Asia,1831,30342957.0,27930630,30008559,...,30263324,33260124,13480571,12209369,12979148,13578114,15854286,17271334,18331635,20688293
4,5,World,NaN,NaN,900,Eastern and South-Eastern Asia,1832,14465509.0,17262816,20822011,...,17926170,19677034,7395672,8889707,11060445,13077356,16213093,18793920,20297350,21732201


--- 
### Step 2: Filter and validate data relevant to analysis and mapping

In [ ]:
# Copy & rename relevant data into a new dataframe
migration = df.rename(columns={
    "Region, development group, country or area of destination": "destination",
    "Location code of destination": "destination_code",
    "Region, development group, country or area of origin": "origin",
    "Location code of origin": "origin_code",
    2024: "population_2024"
})

migration = migration[
    ["destination", "destination_code", "origin", "origin_code", "population_2024"]
].copy()

migration.head()

,destination,destination_code,origin,origin_code,population_2024
0,World,900,World,900,304021813
1,World,900,Sub-Saharan Africa,1834,30661610
2,World,900,Northern Africa and Western Asia,1833,40529326
3,World,900,Central and Southern Asia,1831,53948417
4,World,900,Eastern and South-Eastern Asia,1832,41409235


In [ ]:
# Validate data: missing, duplicates, non-numeric, and negative values
migration.info()

print("\nMissing values:")
print(migration.isna().sum())

print(
    "\nDuplicate origin–destination pairs:",
    migration.duplicated(
        subset=["origin_code", "destination_code"]
    ).sum()
)

population = pd.to_numeric(migration["population_2024"], errors="coerce")

print("Non-numeric values:", population.isna().sum())
print("Negative values:", (population < 0).sum())

<class 'pandas.DataFrame'>
RangeIndex: 28030 entries, 0 to 28029
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   destination       28030 non-null  str  
 1   destination_code  28030 non-null  int64
 2   origin            28030 non-null  str  
 3   origin_code       28030 non-null  int64
 4   population_2024   28030 non-null  int64
dtypes: int64(3), str(2)
memory usage: 1.1 MB

Missing values:
destination         0
destination_code    0
origin              0
origin_code         0
population_2024     0
dtype: int64

Duplicate origin–destination pairs: 0
Non-numeric values: 0
Negative values: 0


In [ ]:
# Filter for records where destination or origin is Singapore
sg_migration = migration[
    (migration["destination"] == "Singapore")| 
    (migration["origin"] == "Singapore")
].copy()

sg_migration.head()

,destination,destination_code,origin,origin_code,population_2024
123,World,900,Singapore,702,269613
402,Sub-Saharan Africa,1834,Singapore,702,1571
628,Northern Africa and Western Asia,1833,Singapore,702,958
808,Central and Southern Asia,1831,Singapore,702,19209
966,Eastern and South-Eastern Asia,1832,Singapore,702,108806


### 📊 Step 3: Summary Statistics
Aggregate totals provide context for analysing and mapping Singapore’s migrant origins and destinations.

In [22]:
# UN codes for Africa, Asia, Europe, Latin America and the Caribbean,
# Northern America, and Oceania
region_codes = [903, 935, 908, 904, 905, 909]

print("📊 Origins of migrants living in Singapore — by region")
display(
    migration.loc[
        (migration["destination"] == "Singapore") &
        migration["origin_code"].isin(region_codes),
        ["origin", "population_2024"]
    ].sort_values("population_2024", ascending=False)
)

print("📊 Destinations of migrants originating from Singapore — by region")
display(
    migration.loc[
        (migration["origin"] == "Singapore") &
        migration["destination_code"].isin(region_codes),
        ["destination", "population_2024"]
    ].sort_values("population_2024", ascending=False)
)

📊 Origins of migrants living in Singapore — by region


,origin,population_2024
12395,ASIA,2659190
12405,NORTHERN AMERICA,29902
12408,OCEANIA,20600


📊 Destinations of migrants originating from Singapore — by region


,destination,population_2024
10251,ASIA,128048
26276,OCEANIA,75921
13967,EUROPE,49805
25649,NORTHERN AMERICA,12878
6200,AFRICA,2496
20982,LATIN AMERICA AND THE CARIBBEAN,465


### Step 4: Process Data for Mapping
Select the top 10 origins of migrants living in Singapore and the top 10 destinations of migrants originating from Singapore, ranked by mid-2024 population.

In [27]:
# Keep only country/area connections
sg_migration = sg_migration[
    (sg_migration["origin_code"] < 900) &
    (sg_migration["destination_code"] < 900)
].copy()

In [28]:
top10 = pd.concat([
    sg_migration[
        sg_migration["destination"] == "Singapore"
    ].nlargest(10, "population_2024").assign(direction="Into Singapore"),

    sg_migration[
        sg_migration["origin"] == "Singapore"
    ].nlargest(10, "population_2024").assign(direction="From Singapore")
], ignore_index=True)

top10

,destination,destination_code,origin,origin_code,population_2024,direction
0,Singapore,702,Malaysia,458,1553249,Into Singapore
1,Singapore,702,China,156,533745,Into Singapore
2,Singapore,702,India,356,377038,Into Singapore
3,Singapore,702,Indonesia,360,180437,Into Singapore
4,Singapore,702,United States of America*,840,26705,Into Singapore
5,Singapore,702,Australia*,36,17222,Into Singapore
6,Singapore,702,"China, Taiwan Province of China",158,11137,Into Singapore
7,Singapore,702,"China, Hong Kong SAR",344,3584,Into Singapore
8,Singapore,702,New Zealand*,554,3378,Into Singapore
9,Singapore,702,Canada,124,3197,Into Singapore


### 📊 Summary Stats for the Top 10
Population counts for each of the top 10 origins and destinations, with percentages of the corresponding total migrant population.

In [35]:
# World totals from the original migration DataFrame
total_in = migration.loc[
    (migration["destination"] == "Singapore") &
    (migration["origin"] == "World"),
    "population_2024"
].iloc[0]

total_out = migration.loc[
    (migration["origin"] == "Singapore") &
    (migration["destination"] == "World"),
    "population_2024"
].iloc[0]

# Aggregate the top 10 in each direction
top10_in = top10.loc[
    top10["direction"] == "Into Singapore", "population_2024"
].sum()

top10_out = top10.loc[
    top10["direction"] == "From Singapore", "population_2024"
].sum()

# Add a percentage column to the top10 DataFrame
# Add each connection's share of its corresponding World total
top10["percentage"] = (
    top10["population_2024"] /
    top10["direction"].map({
        "Into Singapore": total_in,
        "From Singapore": total_out
    }) * 100
).round(2)

# print summary statistics
print(
    f"📊 Top 10 origins: {top10_in:,.0f} people "
    f"({top10_in / total_in:.1%} of all migrants living in Singapore)"
)

print(
    f"📊 Top 10 destinations: {top10_out:,.0f} people "
    f"({top10_out / total_out:.1%} of all migrants originating from Singapore living abroad)"
)

print(
    top10[
        ["direction", "origin", "destination",
         "population_2024", "percentage"]
    ].to_string(
        index=False,
        formatters={
            "population_2024": "{:,.0f}".format,
            "percentage": "{:.2f}%".format
        }
    )
)

📊 Top 10 origins: 2,709,692 people (95.4% of all migrants living in Singapore)
📊 Top 10 destinations: 256,315 people (95.1% of all migrants originating from Singapore living abroad)
     direction                          origin          destination population_2024 percentage
Into Singapore                        Malaysia            Singapore       1,553,249     54.66%
Into Singapore                           China            Singapore         533,745     18.78%
Into Singapore                           India            Singapore         377,038     13.27%
Into Singapore                       Indonesia            Singapore         180,437      6.35%
Into Singapore        United States of America            Singapore          26,705      0.94%
Into Singapore                       Australia            Singapore          17,222      0.61%
Into Singapore China, Taiwan Province of China            Singapore          11,137      0.39%
Into Singapore            China, Hong Kong SAR            

In [36]:
### Export the filtered data to a CSV file for mapping

# remove asterisks
for column in ["origin", "destination"]:
    top10[column] = (
        top10[column]
        .str.replace("*", "", regex=False)
        .str.strip()
    )


# Export counts and percentages
top10.to_csv("data/sg_migration_top10_2024.csv", index=False)
